In [4]:
# ==========================================
# 📰 FAKE NEWS DETECTOR (FIXED & COMPLETE)
# ==========================================

import os
import time

# 🛑 STEP 0: CLEANUP (Fixes the "Tunnel already online" error)
print("🧹 Cleaning up old processes...")
os.system("pkill streamlit")  # Kill any running Streamlit
os.system("pkill ngrok")      # Kill any running Ngrok
time.sleep(3)                 # Wait for system to release ports
print("✅ Cleanup complete.")

# 1️⃣ Install required packages
print("📦 Installing dependencies (this may take a minute)...")
!pip install streamlit pyngrok transformers torch pandas --quiet

# 2️⃣ Import libraries and configure ngrok
from pyngrok import ngrok, conf

# Set your auth token
conf.get_default().auth_token = "33eEQATYeFwXNMBlPYBjYqGVq9y_5FTbXiK9V4YT8bUATN2RB"

# 3️⃣ Write Streamlit app to app.py
app_code = r'''
import streamlit as st
from transformers import pipeline
import torch
import pandas as pd

st.set_page_config(page_title="Fake News Detector", page_icon="📰", layout="centered")
st.title("📰 Fake News Detector for Students")
st.write("Analyze any news article or social media post to detect reliability and get a short summary.")

# =====================
# Sidebar
# =====================
st.sidebar.header("⚙️ Options")
max_len = st.sidebar.slider("Max summary length", 40, 200, 80)
show_summary = st.sidebar.checkbox("Show summary", True)
st.sidebar.markdown("---")
st.sidebar.caption("Developed by **Keerthikeshan K** | © 2026")

# =====================
# Load Models
# =====================
@st.cache_resource
def load_models():
    device = 0 if torch.cuda.is_available() else -1

    # Fake News Classifier
    classifier = pipeline(
        "text-classification",
        model="jy46604790/Fake-News-Bert-Detect",
        tokenizer="jy46604790/Fake-News-Bert-Detect",
        device=device
    )

    # Summarizer (BART)
    summarizer = pipeline(
        "text-generation",
        model="facebook/bart-large-cnn",
        device=device
    )

    return classifier, summarizer

# Load models once
with st.spinner("⏳ Loading AI Models... (First run takes 30s)"):
    classifier, summarizer = load_models()

# =====================
# History storage
# =====================
if "history" not in st.session_state:
    st.session_state.history = []

# =====================
# Input
# =====================
text_input = st.text_area("🗞 Paste your news article or post here:", height=250)

# =====================
# Analyze
# =====================
if st.button("🔍 Analyze"):

    if not text_input.strip():
        st.warning("⚠️ Please enter some text first.")

    else:
        with st.spinner("Analyzing..."):

            # Limit text to avoid errors
            text_to_analyze = text_input[:1024]

            # Prediction
            result = classifier(text_to_analyze)[0]
            label = result["label"]
            score = round(result["score"] * 100, 2)

            # Map LABEL_0/1 to readable text (Adjust based on model specifics if needed)
            # Typically for this model: LABEL_0 = Fake, LABEL_1 = Real
            verdict = "FAKE" if label == "LABEL_0" else "REAL"

            # Summary
            summary_text = "Summary not available."
            if show_summary:
                try:
                    gen = summarizer(text_to_analyze, max_length=max_len, do_sample=False)
                    summary_text = gen[0]["generated_text"]
                except Exception as e:
                    summary_text = f"Could not generate summary: {e}"

        # =====================
        # Output
        # =====================
        st.subheader("📊 Credibility Analysis")

        col1, col2 = st.columns(2)
        col1.metric("Verdict", verdict)
        col2.metric("Confidence", f"{score}%")

        if verdict == "FAKE":
            st.error(f"🚫 FAKE News — Confidence: {score}%")
        else:
            st.success(f"✅ REAL News — Confidence: {score}%")

        if show_summary:
            st.subheader("📝 Summary")
            st.info(summary_text)

        # Google verify link
        query = "+".join(text_input[:60].split())
        st.markdown(f"[🔎 Verify on Google News](https://news.google.com/search?q={query})")

        # Save history
        st.session_state.history.append({
            "text": text_input[:100] + "...",
            "label": verdict,
            "score": f"{score}%",
            "summary": summary_text
        })

# =====================
# History section
# =====================
if st.session_state.history:

    st.markdown("---")
    st.subheader("📜 Analysis History")

    df = pd.DataFrame(st.session_state.history)

    st.download_button(
        "⬇️ Download History CSV",
        df.to_csv(index=False),
        file_name="history.csv",
        mime="text/csv"
    )

    if st.button("🗑️ Clear History"):
        st.session_state.history.clear()
        st.experimental_rerun()

    for i, record in enumerate(reversed(st.session_state.history), 1):
        with st.expander(f"Entry #{i} — {record['label']} ({record['score']})"):
            st.caption(f"**Snippet:** {record['text']}")
            st.write(f"**Summary:** {record['summary']}")

st.markdown("---")
st.caption("Made with ❤️ using Streamlit & Transformers")
'''

with open("app.py", "w") as f:
    f.write(app_code)

# 4️⃣ Run Streamlit in the background
print("🚀 Starting Streamlit server...")
os.system("nohup streamlit run app.py --server.port 8501 > /dev/null 2>&1 &")
time.sleep(5)  # Give Streamlit time to start

# 5️⃣ Start ngrok
try:
    # Attempt to open the tunnel
    public_url = ngrok.connect(8501)
    print(f"\n🎉 SUCCESS! Your app is running here: {public_url}")
except Exception as e:
    # Fallback if tunnel is already open
    print(f"\n⚠️ Tunnel connection issue: {e}")
    print("♻️ Checking for existing tunnels...")
    tunnels = ngrok.get_tunnels()
    if tunnels:
        print(f"👉 Use this existing URL: {tunnels[0].public_url}")
    else:
        print("❌ Critical Error: Please go to 'Runtime > Disconnect and Delete Runtime' and try again.")

🧹 Cleaning up old processes...
✅ Cleanup complete.
📦 Installing dependencies (this may take a minute)...
🚀 Starting Streamlit server...

🎉 SUCCESS! Your app is running here: NgrokTunnel: "https://objurgative-sensationistic-georgiann.ngrok-free.dev" -> "http://localhost:8501"
